
# ML-08 — Honest Model vs Week-4 Baseline

**Lane:** Content Review Priority Ranking  
**Decision:** Which pseudonymized content pages should an SEO specialist review first?  
**Observed label:** `trend_direction == "down"`  
**Claim limit:** The model ranks pages associated with an observed decline. It does not prove why a page declined or that an intervention will improve it.

This notebook uses the same held-out rows and the same ranking metrics for the Week-4 rule and the learned models. All random seeds are fixed at `42`.



## 1. Method choice and why

This is a **binary ranking** problem: each page either has the observed `down` label or it does not, but the operational output is a ranked review queue. I start with **Logistic Regression** because its probabilities can rank pages and its behavior is readable. I also test a **Random Forest** as a nonlinear comparison, but I do not reward complexity unless it improves the held-out ranking metrics.

The main metric is **average precision (AP)** because the positive class is the item type we want near the top of the queue. I also report **precision@20**, **precision@50**, and **ROC AUC**. The Week-4 baseline is evaluated with exactly the same held-out rows and metrics.

To prevent leakage, I exclude:

- `trend_direction` and `trend_pct` (the target and its direct numeric source),
- `impressions_last_30d` and `impressions_prev_30d` (their ratio reconstructs `trend_pct`),
- `content_id` and `client_id` as model features.

`client_id` is used only for grouped splitting so that a client never appears in both train and test.


In [1]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42

CANDIDATE_PATHS = [
    Path("content_refresh_anonymized.csv"),
    Path("./content_refresh_anonymized.csv"),
    Path("../content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("/mnt/data/content_refresh_anonymized.csv"),
]

data_path = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find content_refresh_anonymized.csv")

df = pd.read_csv(data_path)
print(f"Loaded: {data_path.resolve()}")
print(f"Shape: {df.shape}")
print(f"scikit-learn version: {sklearn.__version__}")

required = {
    "content_id", "client_id", "trend_direction", "trend_pct",
    "impressions_90d", "ctr", "avg_position"
}
missing = required - set(df.columns)
if missing:
    raise KeyError(f"Missing required columns: {sorted(missing)}")

# The observed binary outcome used for ranking.
y = (df["trend_direction"] == "down").astype(int)
groups = df["client_id"]

forbidden_features = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "impressions_prev_30d",
}
feature_columns = [c for c in df.columns if c not in forbidden_features]
X = df[feature_columns].copy()

print(f"Positive rate: {y.mean():.3f}")
print(f"Features available to the model: {len(feature_columns)}")
print("Excluded leakage/identifier fields:", sorted(forbidden_features))


Loaded: /Users/anel.murat/Desktop/intern/ml/content_refresh_anonymized.csv
Shape: (30000, 44)
scikit-learn version: 1.5.1
Positive rate: 0.542
Features available to the model: 38
Excluded leakage/identifier fields: ['client_id', 'content_id', 'impressions_last_30d', 'impressions_prev_30d', 'trend_direction', 'trend_pct']



## 2. Split design

I use a single reproducible **75/25 grouped holdout split by `client_id`**. This is stricter than a random row split because pages from one client can share measurement patterns. Holding out whole clients better tests whether the ranking logic transfers to unseen clients.

The exact same train/test indices are used for the baseline, Logistic Regression, and Random Forest.


In [2]:

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
df_train, df_test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
assert train_clients.isdisjoint(test_clients), "Client leakage between train and test"

split_summary = pd.DataFrame({
    "split": ["train", "test"],
    "rows": [len(train_idx), len(test_idx)],
    "clients": [len(train_clients), len(test_clients)],
    "positive_rate": [y_train.mean(), y_test.mean()],
})

display(split_summary.round(3))
print("Client overlap:", len(train_clients & test_clients))


,split,rows,clients,positive_rate
0,train,22885,24,0.550
1,test,7115,8,0.517


Client overlap: 0



## 3. Train + compare vs my baseline

The preprocessing is fitted on the training split only:

- numeric columns: median imputation, missingness indicators, and standardization;
- categorical columns: most-frequent imputation and one-hot encoding;
- unseen categories in the test clients are ignored safely.

The Week-4 score is rebuilt without learning from test outcomes. Position-bucket CTR medians and percentile reference distributions come only from the training rows.


In [3]:

numeric_features = X_train.select_dtypes(exclude=["object", "category"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipe, numeric_features),
    ("categorical", categorical_pipe, categorical_features),
])

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1500,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=250,
        min_samples_leaf=8,
        max_features="sqrt",
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}

fitted_models = {}
model_scores = {}
for name, estimator in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator),
    ])
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe
    model_scores[name] = pipe.predict_proba(X_test)[:, 1]

print("Models trained:", ", ".join(fitted_models))


Models trained: Logistic Regression, Random Forest


In [4]:

# Rebuild the Week-4 baseline on the same split.
POSITION_BINS = [0, 3, 5, 10, 20, np.inf]
POSITION_LABELS = ["1-3", "4-5", "6-10", "11-20", "21+"]

baseline_train = df_train[["impressions_90d", "ctr", "avg_position"]].copy()
baseline_test = df_test[["impressions_90d", "ctr", "avg_position"]].copy()

# The starter data stores rate columns as percentage points (0.76 means 0.76%).
# Ranking is unchanged by dividing all CTR values by 100, so the original scale is retained.
baseline_train["position_bucket"] = pd.cut(
    baseline_train["avg_position"], POSITION_BINS,
    labels=POSITION_LABELS, include_lowest=True
)
baseline_test["position_bucket"] = pd.cut(
    baseline_test["avg_position"], POSITION_BINS,
    labels=POSITION_LABELS, include_lowest=True
)

train_bucket_medians = baseline_train.groupby(
    "position_bucket", observed=False
)["ctr"].median()

global_train_ctr = baseline_train["ctr"].median()
train_expected = baseline_train["position_bucket"].map(train_bucket_medians).astype(float).fillna(global_train_ctr)
test_expected = baseline_test["position_bucket"].map(train_bucket_medians).astype(float).fillna(global_train_ctr)

train_ctr_gap = (train_expected - baseline_train["ctr"]).clip(lower=0)
test_ctr_gap = (test_expected - baseline_test["ctr"]).clip(lower=0)

def percentile_against_train(train_values, test_values):
    reference = np.sort(np.asarray(train_values, dtype=float))
    return np.searchsorted(reference, np.asarray(test_values, dtype=float), side="right") / len(reference)

baseline_scores = (
    0.65 * percentile_against_train(train_ctr_gap, test_ctr_gap)
    + 0.35 * percentile_against_train(
        baseline_train["impressions_90d"], baseline_test["impressions_90d"]
    )
)

all_scores = {"Week-4 baseline": baseline_scores, **model_scores}

def precision_at_k(y_true, scores, k):
    k = min(k, len(y_true))
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

def evaluate_ranking(name, scores):
    predictions = (np.asarray(scores) >= 0.5).astype(int)
    row = {
        "method": name,
        "average_precision": average_precision_score(y_test, scores),
        "roc_auc": roc_auc_score(y_test, scores),
        "precision_at_20": precision_at_k(y_test, scores, 20),
        "precision_at_50": precision_at_k(y_test, scores, 50),
    }
    # A 0.5 cutoff is meaningful for model probabilities, but not for the hand-built score.
    if name != "Week-4 baseline":
        row.update({
            "precision_at_0.5": precision_score(y_test, predictions, zero_division=0),
            "recall_at_0.5": recall_score(y_test, predictions, zero_division=0),
            "f1_at_0.5": f1_score(y_test, predictions, zero_division=0),
        })
    else:
        row.update({"precision_at_0.5": np.nan, "recall_at_0.5": np.nan, "f1_at_0.5": np.nan})
    return row

comparison = pd.DataFrame([
    evaluate_ranking(name, scores) for name, scores in all_scores.items()
])
comparison["AP_improvement_vs_baseline"] = (
    comparison["average_precision"]
    - comparison.loc[comparison["method"] == "Week-4 baseline", "average_precision"].iloc[0]
)
comparison = comparison.sort_values("average_precision", ascending=False).reset_index(drop=True)

display(comparison.round(3))

best_model_name = comparison.loc[
    comparison["method"] != "Week-4 baseline", "method"
].iloc[0]
best_scores = all_scores[best_model_name]
print(f"Selected model for interpretation: {best_model_name}")
print(f"Test base rate: {y_test.mean():.3f}")


,method,average_precision,roc_auc,precision_at_20,precision_at_50,precision_at_0.5,recall_at_0.5,f1_at_0.5,AP_improvement_vs_baseline
0,Random Forest,0.608,0.623,0.60,0.66,0.596,0.66,0.626,0.067
1,Logistic Regression,0.596,0.599,0.95,0.68,0.579,0.67,0.621,0.055
2,Week-4 baseline,0.541,0.530,0.70,0.74,NaN,NaN,NaN,0.000


Selected model for interpretation: Random Forest
Test base rate: 0.517



### Result interpretation

I select the learned model with the highest held-out **average precision**, not automatically the most complex model. Precision@K is also inspected because the real decision is a limited review queue. A model may win overall AP while a simpler rule still performs well at one particular queue length; both results should be reported rather than hidden.



## 4. Errors and interpretation

I interpret the selected model with permutation importance on the untouched test split. Each feature is shuffled and the drop in average precision is measured. A large drop means the feature was useful to the ranking, but it is still **predictive association**, not causality.

For errors, I inspect:

1. false positives inside the top-50 queue — pages the model prioritizes that are not labeled down;
2. missed declining pages with the lowest predicted scores;
3. precision by content type, to see whether error rates differ across operational groups.


In [5]:

selected_model = fitted_models[best_model_name]

perm = permutation_importance(
    selected_model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "mean_AP_drop": perm.importances_mean,
    "std_AP_drop": perm.importances_std,
}).sort_values("mean_AP_drop", ascending=False)

print("Top 10 permutation importances:")
display(importance.head(10).round(4))

# Build an error-review table without exposing client identifiers.
review = df_test[["content_id", "content_type", "impressions_90d", "ctr", "avg_position"]].copy()
review["actual_down"] = y_test.to_numpy()
review["model_score"] = np.asarray(best_scores)
review["rank"] = review["model_score"].rank(method="first", ascending=False).astype(int)

false_positives_top50 = review[
    (review["rank"] <= 50) & (review["actual_down"] == 0)
].sort_values("rank").head(3)

missed_declines = review[
    review["actual_down"] == 1
].sort_values("model_score", ascending=True).head(3)

print("Three false positives in the top-50 queue:")
display(false_positives_top50.round(4))
print(
    "Why these are hard: their observable snapshot features resemble declining pages, "
    "but the observed trend label is not down. Unseen query mix, seasonality, or noise may explain the mismatch."
)

print()
print("Three declining pages assigned very low scores:")
display(missed_declines.round(4))
print(
    "Why these are hard: they decline despite weak warning signals in the allowed features. "
    "The excluded last-vs-previous 30-day fields would reveal the answer directly, so honest prediction must accept some misses."
)


Top 10 permutation importances:


,feature,mean_AP_drop,std_AP_drop
18,days_with_impressions,0.0248,0.0028
20,clicks_last_30d,0.0208,0.0013
32,avg_position,0.0104,0.0013
10,impressions_90d,0.0080,0.0013
21,sessions_last_30d,0.0039,0.0008
37,position_tier,0.0036,0.0010
36,impression_tier,0.0025,0.0005
34,scroll_rate,0.0019,0.0005
23,sessions_prev_30d,0.0016,0.0007
31,ctr,0.0016,0.0005


Three false positives in the top-50 queue:


,content_id,content_type,impressions_90d,ctr,avg_position,actual_down,model_score,rank
22042,content_2ba626fea4d6,keyword article,360,0.00,7.2,0,0.9384,2
29456,content_b46c62b14582,keyword article,6240,0.13,31.8,0,0.9207,4
3669,content_5ce1a9d3e4d7,keyword article,8076,0.07,8.1,0,0.9072,8


Why these are hard: their observable snapshot features resemble declining pages, but the observed trend label is not down. Unseen query mix, seasonality, or noise may explain the mismatch.

Three declining pages assigned very low scores:


,content_id,content_type,impressions_90d,ctr,avg_position,actual_down,model_score,rank
1864,content_16f38acf0f26,keyword article,2,0.00,50.0,1,0.1364,7066
1724,content_b72b06434946,keyword article,1061,0.19,41.0,1,0.1373,7064
27271,content_7bc32bc1df59,keyword article,1,0.00,0.0,1,0.1380,7063


Why these are hard: they decline despite weak warning signals in the allowed features. The excluded last-vs-previous 30-day fields would reveal the answer directly, so honest prediction must accept some misses.


In [6]:

# Group-level error check for content types with enough test examples.
top50_ids = set(review.nsmallest(50, "rank").index)
review["selected_top50"] = review.index.isin(top50_ids)

group_error = (
    review.groupby("content_type", dropna=False)
    .agg(
        n=("actual_down", "size"),
        positive_rate=("actual_down", "mean"),
        mean_model_score=("model_score", "mean"),
    )
    .reset_index()
)

def top_k_precision_within_group(group, k=20):
    chosen = group.nlargest(min(k, len(group)), "model_score")
    return chosen["actual_down"].mean()

group_precision = (
    review.groupby("content_type", dropna=False)
    .apply(top_k_precision_within_group, include_groups=False)
    .rename("within_group_precision_at_20")
    .reset_index()
)

group_error = group_error.merge(group_precision, on="content_type", how="left")
group_error = group_error[group_error["n"] >= 30].sort_values("within_group_precision_at_20")

print("Error check by content type (groups with at least 30 test rows):")
display(group_error.round(3))

# Final headline table requested by the assignment.
final_table = comparison[[
    "method",
    "average_precision",
    "precision_at_20",
    "precision_at_50",
    "roc_auc",
    "AP_improvement_vs_baseline",
]].copy()

print("Final model-vs-baseline table:")
display(final_table.round(3))


Error check by content type (groups with at least 30 test rows):


,content_type,n,positive_rate,mean_model_score,within_group_precision_at_20
0,comparison article,697,0.572,0.759,0.6
1,keyword article,6418,0.510,0.514,0.7


Final model-vs-baseline table:


,method,average_precision,precision_at_20,precision_at_50,roc_auc,AP_improvement_vs_baseline
0,Random Forest,0.608,0.60,0.66,0.623,0.067
1,Logistic Regression,0.596,0.95,0.68,0.599,0.055
2,Week-4 baseline,0.541,0.70,0.74,0.530,0.000



## Conclusion

The final choice is based on held-out grouped performance. The learned model is useful only to the extent that it improves the same ranking metrics over the Week-4 rule. The top-feature table describes what the model relies on, while the concrete mistakes show that the score is not a diagnosis: some high-scored pages do not decline, and some declining pages have few visible warning signs.

The honest operational use is therefore **prioritization for human review**, not automatic intervention.



## Self-check

- [x] Every section contains both reasoning and executable code
- [x] The model and Week-4 baseline use the exact same grouped holdout rows
- [x] The comparison reports average precision, precision@20, precision@50, and ROC AUC
- [x] `trend_direction`, `trend_pct`, direct trend-window fields, and IDs are excluded as features
- [x] Client IDs are used only for grouped splitting
- [x] Random seeds are fixed and the library version is printed
- [x] Permutation importance is computed on the held-out test set
- [x] Three false positives and three missed positives are inspected
- [x] No client names, URLs, or private queries are displayed
- [x] The notebook runs top to bottom without errors
